# Video Renaming Pipeline — overview

This notebook implements the video renaming pipeline used to standardize processed camera‑trap videos, stamp capture times, and attach hierarchical tags. It combines conversion, OCR, error scanning and metadata updates so files are consistently named and discoverable.

Quick summary of the pipeline
- Ingest raw video files from each deployment's 01_raw-video folder.
- Convert/normalize container if needed (AVI → MP4) and run a decode/encoding error scan (ffmpeg).
- Extract burned‑in timestamps from the top‑left overlay via OCR on sampled frames (tesseract + image preprocessing).
- Fix/validate OCR results, map them to wall clock times and compute accurate start/end timestamps for each clip.
- Build canonical new filenames: DEPLOYDATE_ANIMAL_CAMERA_START_END.mp4 (e.g. 2012-11-02_mile-001_PD-01_2012-11-02_15-37-05_16-01-32.mp4).
- Optionally perform a safe, dry‑run bulk rename across all 02_processed-video folders; resolve collisions by appending _N.
- Write audit artifacts: per‑deployment batch_summary.csv, decode reports, frame OCR tables and a master rename log.
- Stamp capture time and hierarchical tags (XMP/IPTC) with exiftool once names and times are verified.

What this notebook expects
- ffmpeg / ffprobe on PATH
- tesseract on PATH
- exiftool installed (used for metadata writes)
- Python packages: pytesseract, pillow, numpy, pandas, python-dateutil

Safety notes
- Start with DRY_RUN / preview modes to inspect proposed names and logs before applying renames or metadata writes.
- Review generated master log and batch_summary.csv before committing changes.


In [ ]:
%%bash
# Environment variables for the deployment
export DIR_METADATA="/Volumes/WORK-SSD/Media-Datasets/Unpublished/mile-adult-sese_vdr_argentina_RD-KM/2013-11-09_mile-008_media/00_metadata"
export DIR_PROCESSED="/Volumes/WORK-SSD/Media-Datasets/Unpublished/mile-adult-sese_vdr_argentina_RD-KM/2013-11-09_mile-008_media/02_processed-video"
export DEPLOY_DATE="2012-11-02"
export ANIMAL_ID="mile-001"
export CAMERA_ID="PD-01"
export FPS="2"
export DURATION="20"
export ROI="0,0,420,140"

# Make sure the script is executable
chmod +x live_text_rename.swift

# Run the Swift Live Text renamer on the raw-video directory
./live_text_rename.swift \
  "/Volumes/WORK-SSD/Media-Datasets/Unpublished/mile-adult-sese_vdr_argentina_RD-KM/2013-11-09_mile-008_media/01_raw-video/PD-03"

## Stamp capture time and add hierarchical tags to processed videos

This cell scans every deployment under the root folder and reads each `00_metadata/batch_summary.csv`.  
For each row marked `renamed=true`, it:

1. Sets the video’s **capture time** to the `start` timestamp from the CSV
   - `DateTimeOriginal`, `CreateDate`, `ModifyDate`
   - QuickTime/ISO: `MediaCreateDate`, `MediaModifyDate`, `TrackCreateDate`, `TrackModifyDate`
   - XMP (Bridge/Adobe): `XMP:CreateDate`, `XMP:DateCreated`

2. Adds **tags (keywords)** as both:
   - `XMP-dc:Subject` and `IPTC:Keywords` (flat)
   - `XMP-lr:HierarchicalSubject` (hierarchical; Bridge/Lightroom use `|` separators)

Tag schema (derived from folder/file structure):
- `Cameras/{Camera_manufacturer}/{Camera ID}` (e.g., `Cameras/PD/PD-01`)
- `Datasets/{dataset_id}/{deployment_id}` (e.g., `Datasets/mile-adult-sese_vdr_argentina_RD-KM/2012-11-02_mile-001`)
- `Animals/{species_code}/{animal_id}` (e.g., `Animals/mile/mile-001`)
- `Location/{location_type}` (e.g., `Location/wild`)
- `Places/{Locality_ID}` (e.g., `Places/PeninsulaValdes_AR`)

**Assumptions**
- Dataset ID is constant.
- Deployment ID = deployment folder name without `_media` suffix.
- Processed files live in `02_processed-video/` and match `new_name` in the CSV (`DEPLOYDATE_ANIMAL_CAMERA_START_END.mp4`).
- `animal_id` and `camera_id` are parsed from `new_name` (2nd and 3rd underscore-delimited fields).
- You can tweak `LOCATION_TYPE` and `PLACE_ID` below if needed.

> Requires `exiftool` (`brew install exiftool`). Start with `DRY_RUN = True` to preview changes.


In [ ]:
# Run this in a Jupyter/VS Code notebook cell on macOS
import os, sys, csv, subprocess, shutil
from pathlib import Path
from datetime import datetime
from dateutil import parser as dateparser  # pip install python-dateutil if needed

# ========= CONFIG (edit these two lines if needed) =========
ROOT = Path("/Volumes/WORK-SSD/Media-Datasets/Unpublished/mile-adult-sese_vdr_argentina_RD-KM")
DRY_RUN = False   # set True to preview without writing
# ===========================================================

exiftool = shutil.which("exiftool")
if not exiftool:
    raise SystemExit("❌ exiftool not found. Install with: brew install exiftool")

# find all batch_summary.csv files under ROOT
summary_files = list(ROOT.glob("**/00_metadata/batch_summary.csv"))
if not summary_files:
    raise SystemExit(f"❌ No batch_summary.csv files found under {ROOT}")

print(f"Found {len(summary_files)} batch_summary.csv files:")
for sf in summary_files:
    print("  •", sf)

# helper: format times for tags
def to_exif_dt(local_dt: datetime) -> str:
    """EXIF/QuickTime style: 'YYYY:MM:DD HH:MM:SS' (no TZ)"""
    return local_dt.strftime("%Y:%m:%d %H:%M:%S")

def to_xmp_dt(local_dt: datetime) -> str:
    """XMP allows TZ; Bridge likes this for Capture Time."""
    # ISO 8601 without microseconds, with timezone offset if available
    # If naive, treat as local time (no offset)
    if local_dt.tzinfo is not None and local_dt.utcoffset() is not None:
        return local_dt.replace(microsecond=0).isoformat()
    return local_dt.strftime("%Y-%m-%dT%H:%M:%S")

# walk summaries, apply tags
updated, skipped, errors = 0, 0, 0
log_rows = []

for csv_path in summary_files:
    dep_dir = csv_path.parent.parent  # 00_metadata -> deployment root
    processed_dir = dep_dir / "02_processed-video"

    if not processed_dir.exists():
        print(f"⚠️ processed dir missing: {processed_dir}")
        continue

    with csv_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            new_name = row.get("new_name", "").strip()
            start_str = row.get("start", "").strip()
            renamed = row.get("renamed", "").strip().lower() in ("1","true","yes")

            if not renamed or not new_name or not start_str:
                skipped += 1
                continue

            video_path = processed_dir / new_name
            if not video_path.exists():
                print(f"⚠️ missing video for metadata write: {video_path}")
                errors += 1
                continue

            # parse start time (ISO-8601 string from the Swift summary)
            try:
                start_dt = dateparser.parse(start_str)
            except Exception as e:
                print(f"⚠️ can't parse start '{start_str}' in {csv_path.name}: {e}")
                errors += 1
                continue

            exif_dt = to_exif_dt(start_dt)
            xmp_dt  = to_xmp_dt(start_dt)

            # Build exiftool command. We touch a bunch of common tags so Bridge/QuickTime pick it up:
            # - QuickTime/ISO media times
            # - EXIF-style times
            # - XMP CreateDate/DateCreated (Bridge "Capture Time")
            cmd = [
                exiftool,
                "-overwrite_original",
                # EXIF-style (many tools look here)
                f"-DateTimeOriginal={exif_dt}",
                f"-CreateDate={exif_dt}",
                f"-ModifyDate={exif_dt}",
                # QuickTime/ISO times used by MP4/MOV
                f"-TrackCreateDate={exif_dt}",
                f"-TrackModifyDate={exif_dt}",
                f"-MediaCreateDate={exif_dt}",
                f"-MediaModifyDate={exif_dt}",
                # Apple 'Keys' group (shown by some apps)
                f"-Keys:CreationDate={exif_dt}",
                # XMP (Adobe Bridge "Capture Time" typically prefers XMP)
                f"-XMP:CreateDate={xmp_dt}",
                f"-XMP:DateCreated={xmp_dt}",
                str(video_path)
            ]

            if DRY_RUN:
                print("DRY-RUN:", " ".join(cmd))
                updated += 1
                log_rows.append((str(video_path), start_str, exif_dt, xmp_dt, "DRY_RUN"))
            else:
                res = subprocess.run(cmd, capture_output=True, text=True)
                if res.returncode == 0:
                    updated += 1
                    log_rows.append((str(video_path), start_str, exif_dt, xmp_dt, "OK"))
                else:
                    errors += 1
                    print(f"❌ exiftool failed on {video_path.name}: {res.stderr.strip()}")
                    log_rows.append((str(video_path), start_str, exif_dt, xmp_dt, f"ERROR: {res.stderr.strip()}"))

# write a small audit log next to ROOT
audit_path = ROOT / f"capture_time_update_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
with audit_path.open("w", newline="", encoding="utf-8") as outf:
    w = csv.writer(outf)
    w.writerow(["file", "start_from_summary", "exif_dt_written", "xmp_dt_written", "status"])
    w.writerows(log_rows)

print("\n— Summary —")
print(f"Updated: {updated}  |  Skipped (no data): {skipped}  |  Errors: {errors}")
print(f"Audit log: {audit_path}")


In [ ]:
# Test stamping capture time + tags on a single video
import os, subprocess, shutil
from pathlib import Path
from datetime import datetime
from dateutil import parser as dateparser

# ==== CONFIG ====
# Pick one of your processed videos
TEST_VIDEO = Path("/Volumes/WORK-SSD/Media-Datasets/Unpublished/mile-adult-sese_vdr_argentina_RD-KM/2012-11-02_mile-001_media/02_processed-video/2012-11-02_mile-001_PD-01_2012-11-02_17-41-55_18-10-12.mp4")

# Matching start time (from batch_summary.csv)
START_STR = "2012-11-02T17:41:55"   # ISO 8601 string

# Tag schema constants
DATASET_ID = "mile-adult-sese_vdr_argentina_RD-KM"
LOCATION_TYPE = "wild"
PLACE_ID = "PeninsulaValdes_AR"

# Camera and animal info parsed manually for this test
ANIMAL_ID = "mile-001"
CAMERA_ID = "PD-01"

# Create test output folder
TEST_OUT = TEST_VIDEO.parent.parent / "test-output"
TEST_OUT.mkdir(exist_ok=True)
TEST_COPY = TEST_OUT / TEST_VIDEO.name
shutil.copy2(TEST_VIDEO, TEST_COPY)
print(f"Copied test file to: {TEST_COPY}")

# ==== Build tag lists ====
deployment_dir = TEST_VIDEO.parent.parent
deployment_id = deployment_dir.name.replace("_media", "")
cam_mfr = CAMERA_ID.split("-", 1)[0]
species_code = ANIMAL_ID.split("-", 1)[0]

flat_tags = [
    f"Cameras/{cam_mfr}/{CAMERA_ID}",
    f"Datasets/{DATASET_ID}/{deployment_id}",
    f"Animals/{species_code}/{ANIMAL_ID}",
    f"Location/{LOCATION_TYPE}",
    f"Places/{PLACE_ID}"
]
hier_tags = [t.replace("/", "|") for t in flat_tags]

# ==== Date conversions ====
start_dt = dateparser.parse(START_STR)
exif_dt = start_dt.strftime("%Y:%m:%d %H:%M:%S")
xmp_dt  = start_dt.strftime("%Y-%m-%dT%H:%M:%S")

# ==== Build exiftool command ====
exiftool = shutil.which("exiftool")
if not exiftool:
    raise SystemExit("❌ exiftool not found (brew install exiftool)")

cmd = [
    exiftool,
    "-overwrite_original",
    f"-DateTimeOriginal={exif_dt}",
    f"-CreateDate={exif_dt}",
    f"-ModifyDate={exif_dt}",
    f"-TrackCreateDate={exif_dt}",
    f"-TrackModifyDate={exif_dt}",
    f"-MediaCreateDate={exif_dt}",
    f"-MediaModifyDate={exif_dt}",
    f"-Keys:CreationDate={exif_dt}",
    f"-XMP:CreateDate={xmp_dt}",
    f"-XMP:DateCreated={xmp_dt}",
]
for t in flat_tags:
    cmd.extend([f"-Subject+={t}", f"-Keywords+={t}"])
for ht in hier_tags:
    cmd.append(f"-XMP-lr:HierarchicalSubject+={ht}")
cmd.append(str(TEST_COPY))

# ==== Run ====
print("\nRunning exiftool:")
print(" ".join(cmd))

res = subprocess.run(cmd, capture_output=True, text=True)
print("\nSTDOUT:", res.stdout)
if res.stderr.strip():
    print("STDERR:", res.stderr.strip())

print("\n✅ Test complete — check:")
print(f"  {TEST_COPY}")
print("View tags with:  exiftool -G1 -a -s", TEST_COPY)


In [ ]:
%%bash
# navigate to the folder that has all your processed .mp4 files
# process every processed-video .mp4 under the dataset root (handles different deployments)
BASE="/Volumes/WORK-SSD/Media-Datasets/Unpublished/mile-adult-sese_vdr_argentina_RD-KM"

find "$BASE" -type f -path "*/02_processed-video/*.mp4" -print0 | while IFS= read -r -d '' file; do
    echo "Processing: $file"
    fname=$(basename "$file")

    datetime_raw=$(echo "$fname" | grep -oE '[0-9]{4}-[0-9]{2}-[0-9]{2}_[0-9]{2}-[0-9]{2}-[0-9]{2}' | head -n1)

    if [ -n "$datetime_raw" ]; then
        date_part="${datetime_raw%%_*}"     # e.g. 2018-04-21
        time_part="${datetime_raw##*_}"     # e.g. 13-32-37

        setfile_date=$(echo "$date_part $time_part" | sed -E \
            's/^([0-9]{4})-([0-9]{2})-([0-9]{2}) ([0-9]{2})-([0-9]{2})-([0-9]{2})$/\2\/\3\/\1 \4:\5:\6/')

        touch_date=$(echo "$datetime_raw" | sed -E 's/[-_]//g' | \
            awk '{print substr($0,1,12) "." substr($0,13,2)}')

        echo "Setting file date to '$setfile_date' for '$file'"
        SetFile -d "$setfile_date" "$file" 2>/dev/null || echo "⚠️ SetFile failed (is Xcode command-line tools installed?)"
        touch -t "$touch_date" "$file"
    else
        echo "❌ Could not extract datetime from: $file"
    fi
done

## Batch rename helper (cell beneath)

This bash cell performs a safe, bulk rename of processed video files under the dataset root.

What it does
- Walks every "*/02_processed-video" folder under ROOT.
- For each deployment (parent folder ending with `_media`) it:
    - Detects the single logger subfolder in `01_raw-video` (skips if not exactly one).
    - Builds a new prefix: `DEPLOYMENTID_LOGGERID_`.
    - Rewrites file names that match the old camera timestamp/prefix pattern to use the new prefix.
    - Handles name collisions by appending `_N` to the base name.
    - Records every action in a TSV master log: `rename_master_log_<timestamp>.tsv`.

Files touched
- Only files matching `*.mp4`, `*.mov`, `*.m4v` in each `02_processed-video` folder.

Safety / usage
- Toggle dry-run by setting environment variable `DRY_RUN=1` (preview) or `DRY_RUN=0` (apply). Default in-cell is set via `DRY_RUN=${DRY_RUN=0}`.
- Review the generated master log before applying changes when using dry-run.
- Assumes the folder layout: `<deployment>_media/01_raw-video/<logger>/` and `<deployment>_media/02_processed-video/`.
- Skips folders that don't match the expected layout (e.g., missing `01_raw-video` or multiple logger dirs).

Output
- Prints per-folder actions to stdout and writes a master TSV `deployment\tlogger\told_name\tnew_name` in the working directory.

In [ ]:
%%bash
# ✅ Set DRY_RUN=1 for preview (no renames), 0 to apply
DRY_RUN=${DRY_RUN=1}

# Root dataset folder
ROOT="/Volumes/WORK-SSD/Media-Datasets/Unpublished/mile-adult-sese_vdr_argentina_RD-KM"

set -euo pipefail

echo "🏁 Starting batch rename from root: $ROOT"
echo "🔎 Dry-run mode: $DRY_RUN"
echo

ts="$(date +%Y%m%d-%H%M%S)"
master_log="rename_master_log_${ts}.tsv"
echo -e "deployment\tlogger\told_name\tnew_name" > "$master_log"

find "$ROOT" -type d -path "*/02_processed-video" | while read -r proc_dir; do
  echo "📂 Processing folder: $proc_dir"
  cd "$proc_dir" || continue

  parent_dir="$(basename "$(dirname "$PWD")")"
  if [[ "$parent_dir" != *_media ]]; then
    echo "  ❌ Skipping (not a *_media folder)"
    continue
  fi
  deployment_id="${parent_dir%_media}"

  raw_dir="../01_raw-video"
  if [[ ! -d "$raw_dir" ]]; then
    echo "  ⚠️ Missing 01_raw-video folder, skipping"
    continue
  fi

  # Find single logger folder
  shopt -s nullglob
  logger_dirs=("$raw_dir"/*/)
  shopt -u nullglob
  if (( ${#logger_dirs[@]} != 1 )); then
    echo "  ⚠️ Found ${#logger_dirs[@]} logger dirs, skipping"
    continue
  fi
  logger_id="$(basename "${logger_dirs[0]}")"

  new_prefix="${deployment_id}_${logger_id}_"
  echo "  🧩 Using prefix: $new_prefix"

  changed_any=false
  shopt -s nullglob
  for f in *.mp4 *.mov *.m4v; do
    [[ -e "$f" ]] || continue
    new_name="$(perl -pe 's|^\d{4}-\d{2}-\d{2}_mile-\d{3}_PD-\d{2}_|'"$new_prefix"'|' <<< "$f")"

    if [[ "$new_name" == "$f" ]]; then
      continue
    fi
    changed_any=true

    if [[ "$DRY_RUN" == "1" ]]; then
      echo "  [DRY] $f → $new_name"
      echo -e "${deployment_id}\t${logger_id}\t${f}\t${new_name}" >> "$master_log"
    else
      target="$new_name"
      if [[ -e "$target" ]]; then
        base="${new_name%.*}"
        ext="${new_name##*.}"
        n=1
        while [[ -e "${base}_$n.$ext" ]]; do ((n++)); done
        target="${base}_$n.$ext"
      fi
      echo "  mv $f → $target"
      mv -v -- "$f" "$target"
      echo -e "${deployment_id}\t${logger_id}\t${f}\t${target}" >> "$master_log"
    fi
  done
  shopt -u nullglob

  if ! $changed_any; then
    echo "  ✅ No changes needed"
  else
    echo "  ✍️  Changes recorded"
  fi
  echo
done

echo "🏁 All done. Master log: $master_log"
if [[ "$DRY_RUN" == "1" ]]; then
  echo "👀 Review log first, then re-run with DRY_RUN=0 to apply renames."
fi
